In [2]:
import numpy as np
import pandas as pd

print('pandas:', pd.__version__, '| numpy:', np.__version__)

pandas: 2.3.3 | numpy: 2.1.3


In [3]:
# Mount Google Drive




df_rna = pd.read_csv('/kaggle/input/datasets/iamahnaf/rnaseq-filtered-survpath/rnaseq_filtered_survpath.csv')
print('Shape:', df_rna.shape)  # expected: (1073, ~4999)
df_rna.head()

Shape: (1073, 4907)


,patient_id,A2M,AAAS,AADAT,AARS2,AASS,ABAT,ABCA1,ABCA2,ABCA3,...,ZNF292,ZNF365,ZNF639,ZNF707,ZNFX1,ZNRF4,ZPBP,ZW10,ZWINT,ZYX
0,TCGA-3C-AAAU,173.5472,32.6330,0.5864,26.3603,1.6383,12.8127,5.5212,82.3301,143.0760,...,8.2165,1.7961,23.2207,13.0002,41.3931,0.0,0.1276,30.4353,82.3660,193.0483
1,TCGA-3C-AALI,193.2961,38.9292,1.8664,16.3475,2.2227,4.4627,6.9859,38.1435,60.4113,...,9.9795,0.9161,12.7001,14.0755,46.5166,0.0,0.0000,21.3471,74.3577,254.0944
2,TCGA-3C-AALJ,204.7472,27.9897,4.0074,13.9400,2.1806,6.3691,10.7979,47.6396,31.7794,...,3.4923,4.1575,14.1258,17.4336,20.7743,0.0,0.1054,17.4182,176.5328,225.4928
3,TCGA-3C-AALK,261.7484,30.4796,3.7265,13.4879,4.7480,28.4268,11.1177,20.3682,20.5225,...,7.4476,4.2791,11.0752,8.7774,24.9127,0.0,0.0000,18.4196,47.9590,245.2738
4,TCGA-4H-AAAK,290.6471,41.2222,4.8974,23.2666,12.3309,17.5398,13.8918,17.5077,32.9598,...,10.1026,4.0954,18.1065,17.6384,38.0379,0.0,0.0637,21.5292,62.4581,220.1794


In [4]:
%pip install -q pandas==2.2.3 anndata scanpy transformers huggingface_hub

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
nilearn 0.14.1 requires pandas>=2.3.0, but you have pandas 2.2.3 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 6.30.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.
Note: you may need to restart the kernel to use updated packages.


In [5]:
from huggingface_hub import login
login()  # paste your Hugging Face token when prompted

In [9]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "ScientaLab/eva-rna"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModel.from_pretrained(
    model_name, 
    trust_remote_code=True, 
    torch_dtype=torch.float16
).to("cuda:0")
model.eval()

print("Model loaded on:", next(model.parameters()).device)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/319 [00:00<?, ?it/s]

Model loaded on: cuda:0


In [10]:
import anndata as ad
import scanpy as sc

# 1. Keep genes present in eva-rna's symbol_to_ncbi mapper
symbol_mapper = tokenizer.gene_mapper["symbol_to_ncbi"]
matched_genes = [g for g in df_rna.columns if g in symbol_mapper]
print(f"SurvPath genes: {df_rna.shape[1]} | Matched in eva-rna: {len(matched_genes)}")

df_rna_eva = df_rna[matched_genes]

# 2. Build AnnData and apply log2(TPM + 1)
adata = ad.AnnData(
    X=df_rna_eva.to_numpy(dtype="float32"),
    obs=pd.DataFrame(index=df_rna_eva.index),
    var=pd.DataFrame(index=df_rna_eva.columns),
)
sc.pp.log1p(adata, base=2)
print("AnnData ready:", adata.shape)
adata


SurvPath genes: 4907 | Matched in eva-rna: 4881
AnnData ready: (1073, 4881)


/usr/lib/python3.13/functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


AnnData object with n_obs × n_vars = 1073 × 4881
    uns: 'log1p'

In [13]:
import gc
import os
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import torch
from transformers import AutoModel

# 1. Clear the leftover OOM memory from cuda:0
gc.collect()
torch.cuda.empty_cache()

# 2. Load a second copy of the model onto cuda:1
model_1 = AutoModel.from_pretrained(
    model_name,
    trust_remote_code=True,
    dtype=torch.float16
).to("cuda:1").eval()

# 3. Split the 1,073 samples evenly across both GPUs
mid = len(adata) // 2
adata_0, adata_1 = adata[:mid], adata[mid:]

def encode_chunk(m, chunk, dev):
    with torch.no_grad():
        return m.encode_anndata(
            tokenizer,
            chunk,
            species="human",
            batch_size=8,  # Safe for 15 GiB T4 VRAM with 4,881 genes
            device=dev,
        )

# 4. Run both GPUs simultaneously
with ThreadPoolExecutor(max_workers=2) as ex:
    f0 = ex.submit(encode_chunk, model, adata_0, "cuda:0")
    f1 = ex.submit(encode_chunk, model_1, adata_1, "cuda:1")
    embeddings = np.vstack([f0.result(), f1.result()]).astype("float32")

print("Embeddings shape:", embeddings.shape)

# 5. Save to Kaggle working directory
emb_cols = [f"eva_{i}" for i in range(embeddings.shape[1])]
df_eva_emb = pd.DataFrame(embeddings, index=adata.obs_names, columns=emb_cols)

output_path = "/kaggle/working/rna_eva_survpath_embeddings.parquet"
df_eva_emb.to_parquet(output_path)
print("Saved to:", output_path)
df_eva_emb.head()

Loading weights:   0%|          | 0/319 [00:00<?, ?it/s]

/root/.cache/huggingface/modules/transformers_modules/ScientaLab/eva_hyphen_rna/4232edf3c7efb01e4bb828be016f3fc86600f3e6/modeling_eva_rna.py:330: UserWarning: Auto-converted 4881/4881 genes from symbol to NCBI GeneIDs (species: human). Set species='human'/'mouse' explicitly if auto-detection is wrong.
  return encode_from_anndata(
/root/.cache/huggingface/modules/transformers_modules/ScientaLab/eva_hyphen_rna/4232edf3c7efb01e4bb828be016f3fc86600f3e6/modeling_eva_rna.py:330: UserWarning: Auto-converted 4881/4881 genes from symbol to NCBI GeneIDs (species: human). Set species='human'/'mouse' explicitly if auto-detection is wrong.
  return encode_from_anndata(
Encoding samples: 100%|██████████| 67/67 [10:19<00:00,  9.24s/batch]

Encoding samples: 100%|██████████| 68/68 [11:28<00:00, 10.13s/batch]

Embeddings shape: (1073, 256)
Saved to: /kaggle/working/rna_eva_survpath_embeddings.parquet


,eva_0,eva_1,eva_2,eva_3,eva_4,eva_5,eva_6,eva_7,eva_8,eva_9,...,eva_246,eva_247,eva_248,eva_249,eva_250,eva_251,eva_252,eva_253,eva_254,eva_255
0,-2.436997,-0.861412,-1.152588,-2.916792,-0.852379,-1.139276,2.390947,0.368506,-0.319133,1.939269,...,0.894699,-2.793308,0.112229,-1.479375,-0.279101,-0.941031,1.199313,-0.064352,1.145598,-1.853099
1,-2.302951,-0.462879,-1.107872,-2.857016,-0.807671,-1.146425,2.502779,-0.495019,-0.657192,1.996245,...,0.621758,-2.106188,-0.348724,-1.315237,0.112782,-0.906523,0.745501,0.029421,1.064512,-1.412543
2,-2.324708,-0.435692,-1.163731,-2.772528,-0.569428,-1.393091,2.433733,-0.347658,-0.548404,2.033797,...,0.900078,-2.071475,-0.273719,-1.397091,-0.131953,-0.790709,0.885062,0.337000,0.989561,-1.492131
3,-2.761796,-0.298790,-1.268128,-2.722769,-0.561944,-1.017221,2.458315,-0.667970,-0.516712,2.004889,...,0.767235,-2.002505,-0.480685,-1.350260,0.191548,-0.593409,0.984111,0.057713,0.522435,-1.317443
4,-2.766755,-0.317732,-1.012208,-2.751722,-0.530083,-1.155645,2.492723,-0.600458,-0.557628,2.249251,...,0.801537,-2.341372,-0.460462,-1.179693,0.379704,-0.393950,0.995897,-0.068930,0.499454,-1.467855
